In [2]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.2/720.2 kB 14.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.3/622.3 kB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 16.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [openmeteo-requests]iquests]uture]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [requests-cache]


In [6]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 53.5507,
	"longitude": 9.993,
	"start_date": "2025-01-01",
	"end_date": "2026-03-31",
	"hourly": ["temperature_2m", "precipitation", "wind_speed_10m", "wind_direction_10m", "cloud_cover"],
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(2).ValuesAsNumpy()
hourly_wind_direction_10m = hourly.Variables(3).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(4).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
hourly_data["cloud_cover"] = hourly_cloud_cover

data_weather = pd.DataFrame(data = hourly_data)
data_weather


Coordinates: 53.560001373291016°N 10.0°E
Elevation: 11.0 m asl
Timezone difference to GMT+0: 0s


,date,temperature_2m,precipitation,wind_speed_10m,wind_direction_10m,cloud_cover
0,2025-01-01 00:00:00+00:00,5.0715,0.0,24.967497,227.337341,100.0
1,2025-01-01 01:00:00+00:00,5.3715,0.0,22.406927,226.301880,100.0
2,2025-01-01 02:00:00+00:00,5.6715,0.0,24.798225,230.300964,100.0
3,2025-01-01 03:00:00+00:00,6.2215,0.0,25.212852,226.735672,100.0
4,2025-01-01 04:00:00+00:00,6.5215,0.0,27.804029,228.674606,100.0
...,...,...,...,...,...,...
10915,2026-03-31 19:00:00+00:00,7.3215,0.0,11.720751,317.489594,47.0
10916,2026-03-31 20:00:00+00:00,6.5215,0.0,11.200571,315.000092,19.0
10917,2026-03-31 21:00:00+00:00,6.1715,0.0,9.387651,302.471161,10.0
10918,2026-03-31 22:00:00+00:00,5.5715,0.0,8.217153,298.810699,20.0
